# $B^+\to K^+\pi^+\pi^-$ fit with efficiency and background

Fit a non-CP $B^+\to K^+\pi^+\pi^-$ model with efficiency, background shape and a floating background fraction.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BaBarFlatte, DecayChannel, DecayModel, LASS, Minimizer, NonResonant,
    Parameter, PhaseSpaceSample, RealImag, Resonance, enable_x64,
    weighted_resample,
)
from dalitzplotfitter.background import FunctionalBackground
from dalitzplotfitter.efficiency import FunctionalEfficiency

enable_x64()


## 1. Amplitude model and normalization

In [ ]:
channel = DecayChannel("B+", ("K+", "pi+", "pi-"))

truth_xy = {
    "Kstar892": (1.00, 0.00),
    "KpiS": (1.40, -0.60),
    "rho770": (0.65, 0.10),
    "f0_980": (-0.20, 1.00),
    "NR": (-0.50, 0.10),
}
truth = {}
def coefficient(name, fixed=False):
    x, y = truth_xy[name]
    if fixed:
        return RealImag(x, y)
    truth[f"{name}.x"], truth[f"{name}.y"] = x, y
    return RealImag(
        Parameter.coefficient(f"{name}.x", x, owner=name, step=0.01),
        Parameter.coefficient(f"{name}.y", y, owner=name, step=0.01),
    )

c = {name: coefficient(name, fixed=(name == "Kstar892")) for name in truth_xy}
components = [
    Resonance("Kstar892", (0,2), c["Kstar892"], mass=0.8958, width=0.0474, spin=1, resonance_radius=4.0, parent_radius=4.0),
    Resonance("KpiS", (0,2), c["KpiS"], lineshape=LASS(2.07, 3.32, 1.8), mass=1.425, width=0.270, spin=0, resonance_radius=4.0, parent_radius=4.0),
    Resonance("rho770", (1,2), c["rho770"], mass=0.7753, width=0.1491, spin=1, resonance_radius=4.0, parent_radius=4.0),
    Resonance("f0_980", (1,2), c["f0_980"], lineshape=BaBarFlatte(), mass=0.965, width=0.0, spin=0, resonance_radius=4.0, parent_radius=4.0),
    NonResonant(c["NR"]),
]
model = DecayModel(
    channel, components,
    normalization_method="square-dalitz",
    normalization_resolution=350,
    normalization_pair=(0, 2),
)
norm = model.normalization_sample
print("free parameters:", len(model.parameters))
print("normalization points:", norm.size)
model.print_fit_fractions(truth, normalization_sample=norm, include_interference=True)


## 2. Efficiency and background models

In [ ]:
s12_min = (channel.daughter_masses[0] + channel.daughter_masses[1])**2
s12_max = (channel.parent_mass - channel.daughter_masses[2])**2
s13_min = (channel.daughter_masses[0] + channel.daughter_masses[2])**2
s13_max = (channel.parent_mass - channel.daughter_masses[1])**2

def scaled(data, key, low, high):
    return jnp.clip((data[key]-low)/(high-low), 0.0, 1.0)

efficiency = FunctionalEfficiency(lambda data:
    0.55
    + 0.30*scaled(data, "s12", s12_min, s12_max)
    + 0.10*jnp.cos(jnp.pi*scaled(data, "s13", s13_min, s13_max))
)
background = FunctionalBackground(lambda data:
    0.50
    + 1.20*scaled(data, "s12", s12_min, s12_max)
    + 0.40*scaled(data, "s13", s13_min, s13_max)
)

eff_norm = efficiency(norm.as_dict())
bkg_norm = jnp.mean(norm.weights * background(norm.as_dict()))
print("efficiency range:", float(eff_norm.min()), float(eff_norm.max()))
print("background normalization:", float(bkg_norm))


## 3. Generate signal plus background pseudo-data

In [ ]:
N_POOL = 250_000
N_DATA = 40_000
BACKGROUND_FRACTION_TRUE = 0.18
pool = model.generate_phase_space(N_POOL, seed=2028)
pool_cache = model.prepare_cache(pool, norm)

signal_weights = pool.weights * efficiency(pool.as_dict()) * pool_cache.intensity(truth)
background_weights = pool.weights * background(pool.as_dict())
n_background = int(round(N_DATA*BACKGROUND_FRACTION_TRUE))
n_signal = N_DATA - n_background
signal_data = weighted_resample(jax.random.key(2029), pool, signal_weights, n_signal, replace=True)
background_data = weighted_resample(jax.random.key(2030), pool, background_weights, n_background, replace=True)

def merge(first, second):
    def joined(name):
        a, b = getattr(first, name), getattr(second, name)
        return None if a is None else jnp.concatenate((a, b))
    return PhaseSpaceSample(
        s12=joined("s12"), s13=joined("s13"), s23=joined("s23"),
        weights=jnp.ones((first.size+second.size,)),
        p1=joined("p1"), p2=joined("p2"), p3=joined("p3"),
    )

data = merge(signal_data, background_data)
print("signal/background/total:", n_signal, n_background, data.size)
fig, ax = plt.subplots(figsize=(7, 5.5))
h = ax.hist2d(np.asarray(data.s12), np.asarray(data.s13), bins=90)
fig.colorbar(h[3], ax=ax, label="events")
ax.set(xlabel=r"$s_{12}$ [GeV$^2$]", ylabel=r"$s_{13}$ [GeV$^2$]")
plt.show()


## Efficiency and background maps

These maps show the efficiency function and normalized background density separately. They are bin averages of the functions, not event populations.

In [ ]:
def binned_function_map(values, bins=70):
    x = np.asarray(pool.s12)
    y = np.asarray(pool.s13)
    values = np.asarray(values)
    total, x_edges, y_edges = np.histogram2d(x, y, bins=bins, weights=values)
    entries, _, _ = np.histogram2d(x, y, bins=(x_edges, y_edges))
    average = np.divide(
        total, entries,
        out=np.full_like(total, np.nan, dtype=float),
        where=entries > 0,
    )
    return average, x_edges, y_edges

efficiency_map, x_edges, y_edges = binned_function_map(
    efficiency(pool.as_dict())
)
background_map, _, _ = binned_function_map(
    background(pool.as_dict()) / bkg_norm
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), constrained_layout=True)
for axis, values, title, colorbar_label in (
    (axes[0], efficiency_map, "Efficiency only", "relative efficiency"),
    (axes[1], background_map, "Background only", "normalized background density"),
):
    image = axis.pcolormesh(x_edges, y_edges, values.T, shading="auto")
    fig.colorbar(image, ax=axis, label=colorbar_label)
    axis.set(
        xlabel=r"$s_{12}$ [GeV$^2$]",
        ylabel=r"$s_{13}$ [GeV$^2$]",
        title=title,
    )
plt.show()


## 4. Fit the complete mixture likelihood

In [ ]:
cache = model.prepare_cache(
    data, norm, efficiency_normalization=eff_norm
)
eff_data = efficiency(data.as_dict())
bkg_data = background(data.as_dict()) / bkg_norm
background_fraction = Parameter(
    "background_fraction", 0.12, bounds=(0.001, 0.50), step=0.01
)
fit_parameters = (*model.parameters, background_fraction)

def nll(values):
    signal_pdf = eff_data * cache.intensity(values) / cache.normalization(values)
    fraction = values["background_fraction"]
    total_pdf = (1.0-fraction)*signal_pdf + fraction*bkg_data
    return -jnp.sum(jnp.log(jnp.clip(total_pdf, min=1e-300)))

rng = np.random.default_rng(314159)
start = {
    parameter.name: truth[parameter.name] + rng.normal(0.0, 0.12)
    for parameter in model.parameters if not parameter.fixed
}
start["background_fraction"] = 0.12
result = Minimizer(nll, fit_parameters, verbose=1).fit(
    start_values=start, simplex=True, ncall=40_000
)
fit_values = {
    parameter.name: float(result.values[parameter.name])
    for parameter in model.parameters if not parameter.fixed
}
print("valid:", result.valid, "NLL:", result.fval, "EDM:", result.fmin.edm)
print("background fraction generated/fitted:",
      BACKGROUND_FRACTION_TRUE, float(result.values["background_fraction"]))
print(f"{'parameter':18s} {'generated':>11s} {'fitted':>11s} {'error':>11s} {'pull':>9s}")
for parameter in model.parameters:
    if parameter.fixed:
        continue
    name = parameter.name
    fitted = float(result.values[name]); error = float(result.errors[name])
    pull = (fitted-truth[name])/error
    print(f"{name:18s} {truth[name]:11.5f} {fitted:11.5f} {error:11.5f} {pull:9.3f}")

model.print_fit_fractions(fit_values, normalization_sample=norm)
model.print_fit_fractions(
    fit_values, normalization_sample=norm, efficiency=efficiency
)


## Fit projections

Compare the toy data with the injected signal-plus-background model and the fitted mixture model.

In [ ]:
projection_cache = model.prepare_cache(
    pool, norm, efficiency_normalization=eff_norm
)
efficiency_pool = efficiency(pool.as_dict())
background_pool = background(pool.as_dict()) / bkg_norm

def mixture_projection(values, background_fraction_value, variable, bins):
    signal = (
        pool.weights * efficiency_pool * projection_cache.intensity(values)
        / projection_cache.normalization(values)
    )
    background_component = pool.weights * background_pool
    mixture = (
        (1.0-background_fraction_value)*signal
        + background_fraction_value*background_component
    )
    histogram, _ = np.histogram(
        np.asarray(getattr(pool, variable)),
        bins=bins,
        weights=np.asarray(mixture),
    )
    return histogram

fitted_background_fraction = float(result.values["background_fraction"])
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
for axis, variable, label in zip(
    axes,
    ("s12", "s13"),
    (r"$s_{12}$ [GeV$^2$]", r"$s_{13}$ [GeV$^2$]"),
):
    observed = np.asarray(getattr(data, variable))
    bins = np.linspace(observed.min(), observed.max(), 70)
    centers = 0.5*(bins[:-1] + bins[1:])
    data_hist, _ = np.histogram(observed, bins=bins)
    generated_hist = mixture_projection(
        truth, BACKGROUND_FRACTION_TRUE, variable, bins
    )
    fitted_hist = mixture_projection(
        fit_values, fitted_background_fraction, variable, bins
    )
    generated_hist *= data_hist.sum()/generated_hist.sum()
    fitted_hist *= data_hist.sum()/fitted_hist.sum()

    axis.errorbar(
        centers, data_hist, yerr=np.sqrt(np.maximum(data_hist, 1.0)),
        fmt=".", color="black", label="toy data",
    )
    axis.step(centers, generated_hist, where="mid", linestyle="--", label="generated model")
    axis.step(centers, fitted_hist, where="mid", label="fitted model")
    axis.set(xlabel=label, ylabel="events / bin")
    axis.legend()
plt.show()


## Interpretation

A single pseudoexperiment is a workflow and closure demonstration. Bias and coverage require an ensemble of statistically independent toys.